# Delta Analysis: Semantic Mutability of Verbs vs Nouns

Loads `output/verb_noun_deltas.csv` (produced by `compute_deltas.py`) and `data/iRT_AJT.csv`.

**Research Questions**
1. Is |Δverb| > |Δnoun|? (mutability hypothesis)
2. Does Δ predict RT? Does it interact with condition?
3. How does pool size affect the stability of X[word]?

## Roadmap
1. Load data, merge, sanity-check coverage
2. RQ1 — mutability: paired t-test + effect size (Cohen's d)
3. RQ2 — RT regression: Δverb and Δnoun as predictors, interaction with condition
4. Pool-size diagnostics: does cosine drop as more contexts included?
5. Export tidy analysis dataset

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scipy.stats as stats
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

ROOT       = Path('..').resolve()
DELTAS_CSV = ROOT / 'output' / 'verb_noun_deltas.csv'
STIMULI    = ROOT / 'data' / 'iRT_AJT.csv'
OUT_DIR    = ROOT / 'output'
OUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

---
## 1. Load and merge

In [ ]:
if not DELTAS_CSV.exists():
    raise FileNotFoundError(
        f'Delta CSV not found at {DELTAS_CSV}.\n'
        'Run compute_deltas.py first (requires ~10 GB RAM for loading .pt files).'
    )

deltas  = pd.read_csv(DELTAS_CSV)
stimuli = pd.read_csv(STIMULI)

# Lowercase join keys to match compute_deltas.py output
stimuli['verb_lc'] = stimuli['verb'].str.lower()
stimuli['noun_lc'] = stimuli['noun'].str.lower()
deltas['verb_lc']  = deltas['verb'].str.lower()
deltas['noun_lc']  = deltas['noun'].str.lower()

df = pd.merge(
    deltas,
    stimuli.drop(columns=['verb', 'noun', 'iRT_AJT', 'Condition'], errors='ignore'),
    on=['verb_lc', 'noun_lc'],
    how='left'
).drop(columns=['verb_lc', 'noun_lc'])

print(f'Rows: {len(df)}  |  Conditions: {df["condition"].value_counts().to_dict()}')
print(f'verb_delta_cos: {df["verb_delta_cos"].notna().sum()} non-null')
print(f'noun_delta_cos: {df["noun_delta_cos"].notna().sum()} non-null')
df.head()

In [ ]:
# Items with complete data (both deltas available)
df_complete = df.dropna(subset=['verb_delta_cos', 'noun_delta_cos', 'iRT_AJT']).copy()
print(f'Complete cases (both deltas + RT): {len(df_complete)}')
print(df_complete['condition'].value_counts())

---
## 2. RQ1 — Mutability: |Δverb| vs |Δnoun|

Hypothesis: verbs (relational categories) show greater deviation from their prototype embedding than nouns (entity categories), i.e., verb cosine similarity to the global mean is *lower* than for nouns.

Note: a *lower* cosine similarity = more context-shift = greater mutability.

In [ ]:
d = df_complete

# Summary stats
print('=== Delta cosine similarity (higher = more similar to prototype = less mutable) ===')
print(d[['verb_delta_cos', 'noun_delta_cos']]
      .describe().round(4).to_string())

print('\n=== By condition ===')
print(d.groupby('condition')[['verb_delta_cos', 'noun_delta_cos']]
      .mean().round(4).to_string())

In [ ]:
# Paired t-test: verb_delta_cos vs noun_delta_cos (same items)
t, p = stats.ttest_rel(d['verb_delta_cos'], d['noun_delta_cos'])

diff = d['verb_delta_cos'] - d['noun_delta_cos']
cohens_d = diff.mean() / diff.std(ddof=1)

print(f'Paired t-test  t = {t:.3f},  p = {p:.4f}')
print(f"Cohen's d      = {cohens_d:.3f}")
print()
print('Interpretation: positive d => verb cos > noun cos => verbs LESS mutable')
print('                negative d => verb cos < noun cos => verbs MORE mutable (predicted)')

In [ ]:
# Also break down by condition
for cond, grp in d.groupby('condition'):
    t_c, p_c = stats.ttest_rel(grp['verb_delta_cos'], grp['noun_delta_cos'])
    diff_c   = grp['verb_delta_cos'] - grp['noun_delta_cos']
    d_c      = diff_c.mean() / diff_c.std(ddof=1)
    print(f'{cond:>14}  n={len(grp):>3}  t={t_c:+.3f}  p={p_c:.4f}  d={d_c:+.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Paired difference distribution
ax = axes[0]
diff_all = d['verb_delta_cos'] - d['noun_delta_cos']
ax.hist(diff_all, bins=25, color='steelblue', edgecolor='white')
ax.axvline(0, color='black', lw=1.2, ls='--')
ax.axvline(diff_all.mean(), color='tomato', lw=1.5, label=f'mean={diff_all.mean():.3f}')
ax.set(xlabel='verb_cos − noun_cos', ylabel='count', title='Mutability difference\n(negative = verb more mutable)')
ax.legend(fontsize=9)

# 2. Boxplot by condition
ax = axes[1]
conds = ['Collocation', 'Productive']
data_v = [d.loc[d['condition']==c, 'verb_delta_cos'].dropna() for c in conds]
data_n = [d.loc[d['condition']==c, 'noun_delta_cos'].dropna() for c in conds]
pos_v = [1, 3]
pos_n = [2, 4]
bp1 = ax.boxplot(data_v, positions=pos_v, widths=0.6, patch_artist=True,
                 boxprops=dict(facecolor='steelblue', alpha=0.7), medianprops=dict(color='navy'))
bp2 = ax.boxplot(data_n, positions=pos_n, widths=0.6, patch_artist=True,
                 boxprops=dict(facecolor='tomato', alpha=0.7), medianprops=dict(color='darkred'))
ax.set_xticks([1.5, 3.5])
ax.set_xticklabels(conds)
ax.set(ylabel='cosine similarity', title='Delta by condition')
ax.legend([bp1['boxes'][0], bp2['boxes'][0]], ['verb', 'noun'], fontsize=9)

# 3. Scatter: verb_delta vs noun_delta, coloured by condition
ax = axes[2]
colours = {'Collocation': 'steelblue', 'Productive': 'tomato'}
for cond, grp in d.groupby('condition'):
    ax.scatter(grp['noun_delta_cos'], grp['verb_delta_cos'],
               c=colours[cond], alpha=0.5, s=20, label=cond)
lims = [min(d['verb_delta_cos'].min(), d['noun_delta_cos'].min()) - 0.01,
        max(d['verb_delta_cos'].max(), d['noun_delta_cos'].max()) + 0.01]
ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5)
ax.set(xlabel='noun_delta_cos', ylabel='verb_delta_cos', title='Verb vs noun cosine')
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(OUT_DIR / 'rq1_mutability.png', bbox_inches='tight')
plt.show()
print('Saved rq1_mutability.png')

---
## 3. RQ2 — Does Δ predict RT?

Hypothesis (from pilot with 10 nouns): as cosine similarity decreases (more context shift), RT increases — but only for collocations, not productive items.

We fit linear mixed models (via OLS on item-level means here; trial-level models need `data/experiment_data_anonymised.csv`).

Models:
- M0: `iRT_AJT ~ verb_delta_cos`
- M1: `iRT_AJT ~ noun_delta_cos`
- M2: `iRT_AJT ~ verb_delta_cos * condition`
- M3: `iRT_AJT ~ noun_delta_cos * condition`
- M4: `iRT_AJT ~ verb_delta_cos + noun_delta_cos + condition` (additive)
- M5: `iRT_AJT ~ (verb_delta_cos + noun_delta_cos) * condition` (full interaction)

In [ ]:
d_reg = df_complete.copy()

# Treatment-code condition (Productive = reference)
d_reg['cond_bin'] = (d_reg['condition'] == 'Collocation').astype(int)

models = {
    'M0 verb_delta':           'iRT_AJT ~ verb_delta_cos',
    'M1 noun_delta':           'iRT_AJT ~ noun_delta_cos',
    'M2 verb*cond':            'iRT_AJT ~ verb_delta_cos * C(condition, Treatment("Productive"))',
    'M3 noun*cond':            'iRT_AJT ~ noun_delta_cos * C(condition, Treatment("Productive"))',
    'M4 additive':             'iRT_AJT ~ verb_delta_cos + noun_delta_cos + cond_bin',
    'M5 full interaction':     'iRT_AJT ~ (verb_delta_cos + noun_delta_cos) * cond_bin',
}

results = {}
for name, formula in models.items():
    fit = smf.ols(formula, data=d_reg).fit()
    results[name] = fit
    print(f'\n--- {name} ---')
    print(f'  R² = {fit.rsquared:.4f}  adj-R² = {fit.rsquared_adj:.4f}  AIC = {fit.aic:.1f}')
    coef_df = fit.summary2().tables[1][['Coef.', 'Std.Err.', 't', 'P>|t|']]
    print(coef_df.round(4).to_string())

In [ ]:
# Model comparison table
rows = []
for name, fit in results.items():
    rows.append({'model': name, 'R²': fit.rsquared, 'adj-R²': fit.rsquared_adj, 'AIC': fit.aic, 'n': int(fit.nobs)})
pd.DataFrame(rows).set_index('model').round(4)

In [ ]:
# Visualise M2 and M3: delta ~ RT by condition
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, delta_col, title in zip(
        axes,
        ['verb_delta_cos', 'noun_delta_cos'],
        ['Verb Δ vs RT', 'Noun Δ vs RT']):

    for cond, colour in [('Collocation', 'steelblue'), ('Productive', 'tomato')]:
        grp = d_reg[d_reg['condition'] == cond]
        ax.scatter(grp[delta_col], grp['iRT_AJT'], c=colour, alpha=0.5, s=18, label=cond)
        # regression line per condition
        m, b = np.polyfit(grp[delta_col].dropna(), grp.loc[grp[delta_col].notna(), 'iRT_AJT'], 1)
        xs = np.linspace(grp[delta_col].min(), grp[delta_col].max(), 100)
        ax.plot(xs, m * xs + b, color=colour, lw=1.8)

    ax.set(xlabel=delta_col, ylabel='iRT_AJT (ms)', title=title)
    ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(OUT_DIR / 'rq2_rt_regression.png', bbox_inches='tight')
plt.show()
print('Saved rq2_rt_regression.png')

---
## 4. Pool-size diagnostics

A key concern: larger pools → more stable (closer to corpus-wide) X[word], potentially artefactually lower cosine similarity. We check whether pool size correlates with delta cosine within each condition.

In [ ]:
print('=== Verb pool size ===')
print(df_complete['verb_pool_size'].describe().round(1))

print('\n=== Noun pool size (−1 = from verbs fallback) ===')
print(df_complete['noun_pool_size'].describe().round(1))
print(df_complete['noun_x_source'].value_counts())

In [ ]:
# Correlation: pool size vs delta cosine
for word, pool_col, delta_col in [
        ('verb', 'verb_pool_size', 'verb_delta_cos'),
        ('noun', 'noun_pool_size', 'noun_delta_cos')]:

    sub = df_complete[[pool_col, delta_col]].dropna()
    r, p = stats.pearsonr(sub[pool_col], sub[delta_col])
    print(f'{word:>4} pool_size vs delta_cos:  r = {r:+.3f},  p = {p:.4f}  (n={len(sub)})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, pool_col, delta_col, label in zip(
        axes,
        ['verb_pool_size', 'noun_pool_size'],
        ['verb_delta_cos', 'noun_delta_cos'],
        ['Verb', 'Noun']):

    sub = df_complete[[pool_col, delta_col, 'condition']].dropna()
    for cond, colour in [('Collocation', 'steelblue'), ('Productive', 'tomato')]:
        grp = sub[sub['condition'] == cond]
        ax.scatter(grp[pool_col], grp[delta_col], c=colour, alpha=0.4, s=15, label=cond)

    m, b = np.polyfit(sub[pool_col], sub[delta_col], 1)
    xs = np.linspace(sub[pool_col].min(), sub[pool_col].max(), 100)
    ax.plot(xs, m * xs + b, 'k--', lw=1.2, alpha=0.7)
    r, p = stats.pearsonr(sub[pool_col], sub[delta_col])
    ax.set(xlabel=pool_col, ylabel=delta_col,
           title=f'{label} pool size vs Δ\nr = {r:+.3f}, p = {p:.3f}')
    ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(OUT_DIR / 'rq3_pool_size.png', bbox_inches='tight')
plt.show()
print('Saved rq3_pool_size.png')

---
## 5. Export tidy analysis dataset

In [ ]:
EXPORT_COLS = [
    'item', 'verb', 'noun', 'condition', 'iRT_AJT',
    'verb_delta_cos', 'verb_pool_size',
    'noun_delta_cos', 'noun_pool_size', 'noun_x_source',
    'noun_concreteness', 'snd3', 'snd10', 'snd25', 'snd50',
    'logDice', 'collocation_freq', 'VerbFreq', 'NounFreq',
    'verb_synset_len',
]
existing = [c for c in EXPORT_COLS if c in df_complete.columns]
out = df_complete[existing].copy()

out_path = OUT_DIR / 'analysis_dataset.csv'
out.to_csv(out_path, index=False)
print(f'Saved -> {out_path}  ({len(out)} rows, {len(existing)} columns)')
out.head()

---
## 6. Interpretation checklist

Fill in after running:

- [ ] RQ1: Is verb_delta_cos significantly < noun_delta_cos? (t-test direction and p)
- [ ] RQ1: Does the effect hold within each condition separately?
- [ ] RQ2: Does any model explain meaningful RT variance (adj-R² > 0.05)?
- [ ] RQ2: Is the verb_delta_cos × condition interaction significant? (replication of pilot)
- [ ] RQ2: Is the noun_delta_cos × condition interaction significant?
- [ ] Pool size: Is there a significant negative correlation between pool_size and delta_cos? If so, consider partial-correlation or residualisation.
- [ ] Noun fallback items: do items with `noun_x_source == 'verbs_pt_fallback'` behave differently? (spot-check with groupby)
- [ ] Missing items: confirm the 3 known exclusions (beat odds, dent cans, launder clothes) are absent from df_complete